# Статистика связи GNSS-индексов и GOES X-ray в момент пика вспышки

Тетрадка собирает события через локальный `results_server`, берет момент пика вспышки из имени события (`start_peak_end_class`), затем находит ближайшие по времени значения GOES X-ray (`xrsb`) и GNSS-индекса из `indices_*.csv`.

Перед запуском должен быть открыт сервер результатов, например: `http://127.0.0.1:8001`.

In [3]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
import re
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlparse
from urllib.request import urlopen

import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE_URL = "http://127.0.0.1:8001"  # results_server URL
RESULTS_DIR = Path("../../results")
OUTPUT_DIR = Path("./outputs")
PREFER_LOCAL_FILES = True
READ_TIMEOUT_SECONDS = 10

PRODUCTS = ("roti", "dtec_2_10", "dtec_10_20", "dtec_20_60")
INDEX_COLUMNS = ("day_night_index", "gsflai_index", "isfai_index")
XRAY_COLUMN = "xrsb"
MAX_TIME_DELTA = pd.Timedelta("90s")
PEAK_TIME_SOURCE = "event_name"  # event_name или goes_max

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")


## Загрузка данных с `results_server`

In [4]:
def normalize_base_url(base_url: str) -> str:
    base_url = str(base_url).strip().rstrip("/")
    while base_url.startswith("http://http://"):
        base_url = base_url.removeprefix("http://")
    while base_url.startswith("https://http://"):
        base_url = base_url.removeprefix("https://")
    if not base_url.startswith(("http://", "https://")):
        base_url = "http://" + base_url
    parsed = urlparse(base_url)
    if not parsed.hostname:
        raise ValueError(f"Invalid BASE_URL: {base_url!r}")
    return base_url


BASE_URL = normalize_base_url(BASE_URL)


def read_text(url: str, timeout: int = READ_TIMEOUT_SECONDS) -> str:
    try:
        with urlopen(url, timeout=timeout) as response:
            return response.read().decode("utf-8")
    except URLError as exc:
        raise RuntimeError(
            f"Cannot open {url}. Check that results_server is running and BASE_URL={BASE_URL!r}. "
            "From the repository root, run: python results_server.py --port 8001 --directory ./results"
        ) from exc


def url_join(base_url: str, *parts: str) -> str:
    base = normalize_base_url(base_url)
    encoded = "/".join(quote(str(part).strip("/")) for part in parts if str(part).strip("/"))
    return f"{base}/{encoded}"


def event_file_path(event: dict, *parts: str) -> Path:
    return RESULTS_DIR / event["path"] / Path(*parts)


def load_events(base_url: str = BASE_URL) -> list[dict]:
    return json.loads(read_text(f"{normalize_base_url(base_url)}/api/events"))


def read_csv_url(url: str) -> pd.DataFrame:
    return pd.read_csv(StringIO(read_text(url)))


def read_event_csv(event: dict, local_parts: tuple[str, ...], fallback_url: str) -> pd.DataFrame:
    local_path = event_file_path(event, *local_parts)
    if PREFER_LOCAL_FILES and local_path.exists():
        return pd.read_csv(local_path)
    return read_csv_url(fallback_url)


print(f"Using results_server: {BASE_URL}")
print(f"Using local results dir: {RESULTS_DIR.resolve()}")
print(f"Saving analysis outputs to: {OUTPUT_DIR.resolve()}")
events = load_events()
print(f"Событий в API: {len(events)}")
pd.DataFrame(events).head()


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

## Функции извлечения пика GOES и индекса в момент пика

In [ ]:
def normalize_time_column(df: pd.DataFrame, preferred: str = "time") -> pd.DataFrame:
    df = df.copy()
    if preferred not in df.columns:
        first_column = df.columns[0]
        df = df.rename(columns={first_column: preferred})
    df[preferred] = pd.to_datetime(df[preferred], utc=True, errors="coerce")
    return df.dropna(subset=[preferred]).sort_values(preferred)


def load_goes(event: dict, base_url: str = BASE_URL) -> pd.DataFrame:
    url = url_join(base_url, event["path"], "goes_xray", "goes_xray.csv")
    df = normalize_time_column(read_event_csv(event, ("goes_xray", "goes_xray.csv"), url), preferred="time")
    for column in ("xrsa", "xrsb"):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def load_indices(event: dict, product: str, base_url: str = BASE_URL) -> pd.DataFrame:
    url = url_join(base_url, event["path"], "indices", f"indices_{product}.csv")
    df = normalize_time_column(read_event_csv(event, ("indices", f"indices_{product}.csv"), url), preferred="time")
    for column in INDEX_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def parse_event_peak_time(event: dict) -> pd.Timestamp | None:
    timestamps = re.findall(r"\d{8}T\d{6}", event.get("name", ""))
    if len(timestamps) < 2:
        return None
    return pd.to_datetime(timestamps[1], format="%Y%m%dT%H%M%S", utc=True, errors="coerce")


def goes_peak(goes: pd.DataFrame, xray_column: str = XRAY_COLUMN) -> pd.Series:
    if xray_column not in goes.columns:
        raise ValueError(f"В GOES CSV нет колонки {xray_column!r}")
    values = pd.to_numeric(goes[xray_column], errors="coerce")
    if values.dropna().empty:
        raise ValueError(f"В колонке {xray_column!r} нет численных данных")
    return goes.loc[values.idxmax()]


def nearest_goes_row(goes: pd.DataFrame, target_time: pd.Timestamp, tolerance: pd.Timedelta = MAX_TIME_DELTA) -> pd.Series | None:
    if goes.empty:
        return None
    deltas = (goes["time"] - target_time).abs()
    nearest_idx = deltas.idxmin()
    if deltas.loc[nearest_idx] > tolerance:
        return None
    return goes.loc[nearest_idx]


def nearest_index_row(indices: pd.DataFrame, target_time: pd.Timestamp, tolerance: pd.Timedelta = MAX_TIME_DELTA) -> pd.Series | None:
    if indices.empty:
        return None
    deltas = (indices["time"] - target_time).abs()
    nearest_idx = deltas.idxmin()
    if deltas.loc[nearest_idx] > tolerance:
        return None
    return indices.loc[nearest_idx]


## Сбор таблицы статистики

Одна строка соответствует паре `событие + продукт`. Для каждого продукта записываются значения всех доступных индексных колонок в ближайший момент к пику `xrsb`.

In [ ]:
rows: list[dict] = []
errors: list[dict] = []

usable_events = [event for event in events if event.get("sources", {}).get("goes_xray") and any(event.get("indices", {}).values())]
print(f"Событий с GOES и хотя бы одним индексом: {len(usable_events)}")

for event_idx, event in enumerate(usable_events, 1):
    if event_idx == 1 or event_idx % 10 == 0 or event_idx == len(usable_events):
        print(f"[{event_idx}/{len(usable_events)}] {event.get('name')}")
    try:
        goes = load_goes(event)
        flare_peak_time = parse_event_peak_time(event) if PEAK_TIME_SOURCE == "event_name" else None
        if flare_peak_time is not None and pd.notna(flare_peak_time):
            peak = nearest_goes_row(goes, flare_peak_time)
            if peak is None:
                raise ValueError("нет GOES-измерения рядом с peak time из имени события")
            peak_source = "event_name"
        else:
            peak = goes_peak(goes)
            flare_peak_time = peak["time"]
            peak_source = "goes_max"
    except (HTTPError, URLError, TimeoutError, ValueError, OSError) as exc:
        errors.append({"event": event.get("name"), "stage": "goes", "error": str(exc)})
        continue

    for product in PRODUCTS:
        if not event.get("indices", {}).get(product):
            continue
        try:
            indices = load_indices(event, product)
            nearest = nearest_index_row(indices, flare_peak_time)
            if nearest is None:
                errors.append({"event": event.get("name"), "product": product, "stage": "nearest", "error": "нет индекса рядом с flare peak"})
                continue

            row = {
                "event": event["name"],
                "event_path": event["path"],
                "flare_class": event.get("class"),
                "product": product,
                "flare_peak_time": flare_peak_time,
                "goes_time": peak["time"],
                "peak_source": peak_source,
                "index_time": nearest["time"],
                "index_time_delta_seconds": abs((nearest["time"] - flare_peak_time).total_seconds()),
                "goes_time_delta_seconds": abs((peak["time"] - flare_peak_time).total_seconds()),
                "xray_column": XRAY_COLUMN,
                "xray_at_flare_peak": float(peak[XRAY_COLUMN]),
            }
            for column in INDEX_COLUMNS:
                row[column] = float(nearest[column]) if column in nearest.index and pd.notna(nearest[column]) else np.nan
            rows.append(row)
        except (HTTPError, URLError, TimeoutError, ValueError, OSError) as exc:
            errors.append({"event": event.get("name"), "product": product, "stage": "indices", "error": str(exc)})

stats = pd.DataFrame(rows)
errors_df = pd.DataFrame(errors)
print(f"Строк статистики: {len(stats)}")
print(f"Ошибок/пропусков: {len(errors_df)}")
stats.head()

In [ ]:
if not errors_df.empty:
    display(errors_df.head(20))

## Графики зависимости индекса от X-ray в момент пика вспышки

In [ ]:
def plot_index_vs_xray(stats: pd.DataFrame, index_column: str, use_log_x: bool = True) -> None:
    data = stats.dropna(subset=["xray_at_flare_peak", index_column]).copy()
    if data.empty:
        print(f"Нет данных для {index_column}")
        return

    products = [product for product in PRODUCTS if product in set(data["product"])]
    cols = 2
    rows_count = math.ceil(len(products) / cols)
    fig, axes = plt.subplots(rows_count, cols, figsize=(13, 4.8 * rows_count), squeeze=False)
    axes_flat = axes.ravel()

    for ax, product in zip(axes_flat, products):
        subset = data[data["product"] == product]
        ax.scatter(subset["xray_at_flare_peak"], subset[index_column], s=52, alpha=0.82)
        ax.set_title(product)
        ax.set_xlabel(f"GOES {XRAY_COLUMN} at flare peak, W/m²")
        ax.set_ylabel(index_column)
        if use_log_x:
            ax.set_xscale("log")
        if len(subset) >= 2:
            corr = subset[["xray_at_flare_peak", index_column]].corr(method="spearman").iloc[0, 1]
            ax.text(0.03, 0.96, f"n={len(subset)}\nSpearman r={corr:.2f}", transform=ax.transAxes, va="top")
        else:
            ax.text(0.03, 0.96, f"n={len(subset)}", transform=ax.transAxes, va="top")

    for ax in axes_flat[len(products):]:
        ax.axis("off")

    fig.suptitle(f"{index_column} и GOES {XRAY_COLUMN} в момент пика вспышки", fontsize=15)
    fig.tight_layout()
    plt.show()


for index_column in INDEX_COLUMNS:
    plot_index_vs_xray(stats, index_column)

## Общий график с разделением по продукту

In [ ]:
index_column = "gsflai_index"
data = stats.dropna(subset=["xray_at_flare_peak", index_column]).copy()

fig, ax = plt.subplots(figsize=(10, 6))
for product, subset in data.groupby("product"):
    ax.scatter(subset["xray_at_flare_peak"], subset[index_column], s=58, alpha=0.82, label=product)

ax.set_xscale("log")
ax.set_xlabel(f"GOES {XRAY_COLUMN} at flare peak, W/m²")
ax.set_ylabel(index_column)
ax.set_title(f"{index_column}: индекс и GOES {XRAY_COLUMN} в момент пика вспышки")
ax.legend(title="Product")
fig.tight_layout()
plt.show()

## Таблица корреляций

In [ ]:
corr_rows = []
for product, product_df in stats.groupby("product"):
    for index_column in INDEX_COLUMNS:
        subset = product_df.dropna(subset=["xray_at_flare_peak", index_column])
        if len(subset) < 2:
            corr = np.nan
        else:
            corr = subset[["xray_at_flare_peak", index_column]].corr(method="spearman").iloc[0, 1]
        corr_rows.append({"product": product, "index": index_column, "n": len(subset), "spearman_r": corr})

correlations = pd.DataFrame(corr_rows).sort_values(["index", "product"])
correlations

## Сохранение результата

In [ ]:
stats_path = OUTPUT_DIR / "xray_index_peak_statistics.csv"
correlations_path = OUTPUT_DIR / "xray_index_peak_correlations.csv"

stats.to_csv(stats_path, index=False)
correlations.to_csv(correlations_path, index=False)
print(f"Сохранено: {stats_path}")
print(f"Сохранено: {correlations_path}")